# 1D linear shallow water equations in JAX

The shallow water equations are a depth-averaged approximation to the Navier-Stokes equations. The standard 2D formulation makes use of a horizontal velocity $\mathbf{u}:\Omega\times(0,T)\rightarrow\mathbb{R}^2$ and a free surface elevation field $\eta:\Omega\times(0,T)\rightarrow\mathbb{R}$, where we have a spatial domain $\Omega\subset\mathbb{R}^2$ and time period $(0,T)$ for some $T>0$.

In this mini-project, we apply several simplifications:

1. Consider a 1D spatial domain rather than a 2D spatial domain.
2. Do not consider Coriolis forces due to planetary rotation.
3. Do not consider nonlinear terms.

The non-rotational shallow water equations are given by
$$
\begin{align}
\frac{\partial u}{\partial t}+\nabla\cdot(\nabla\mathbf{u})+g\nabla\eta&=\nabla\cdot(\nu(\nabla\mathbf{u}+(\nabla\mathbf{u})^T)),\\
\frac{\partial\eta}{\partial t}+\nabla\cdot(h\mathbf{u})&=0,
\end{align}
$$
where $h=b+\eta$ is the total water depth, $b:\Omega\rightarrow\mathbb{R}$ is the fixed (constant in time) bathymetry, $\nu$ is a viscosity coefficient, and $g$ is the gravitational acceleration constant.

Restricting to the 1D case and linearising about $\overline{u}=0$ and $\overline{\eta}=0$ ('lake at rest'), the advection and viscosity terms are dropped and we are left with
$$
\begin{align}
\frac{\partial u}{\partial t}+g\frac{\partial\eta}{\partial x}&=0,\\
\frac{\partial\eta}{\partial t}+b\frac{\partial u}{\partial x}&=0,
\end{align}
$$
where now $u:\Omega\times(0,T)\rightarrow\mathbb{R}$.

Consider the 1D spatial domain $\Omega=(0,X)$ with $X>0$.

Combining the state as a vector $\mathbf{w}=(u,\eta)$ we have
$$\frac{\partial\mathbf{w}}{\partial t}=-\begin{bmatrix}0&g\\b&0\end{bmatrix}\frac{\partial\mathbf{w}}{\partial x},$$
which can be show to be equivalent to a wave equation with wavespeed $\sqrt{gb}$.
We introduce the notation
$$\underline{\mathbf{A}}=\begin{bmatrix}0&g\\b&0\end{bmatrix}$$
for conciseness.

For initial conditions, assume zero velocity $u(x,0)=0,\:\forall x\in(0,X)$ and spatially varying free surface $\eta(x,0)=\eta_0(x)$.

Assume periodic boundary conditions for simplicity: $\mathbf{w}(0,t)=\mathbf{w}(X,t),\:\forall t\in(0,T)$.

Denoting the matrix by $\underline{\mathbf{A}}$ and applying a $\theta$-method for timestepping gives the approximation
$$\frac{\mathbf{w}^{k+1}-\mathbf{w}^k}{\Delta t}\approx-\underline{\mathbf{A}}\left((1-\theta)\frac{\partial\mathbf{w}^{k+1}}{\partial x}+\theta\frac{\partial\mathbf{w}^k}{\partial x}\right)$$

Applying a central difference for the spatial derivative gives
$$\frac{\mathbf{w}^{k+1}_i-\mathbf{w}^k_i}{\Delta t}\approx-\underline{\mathbf{A}}\left((1-\theta)\frac{\mathbf{w}^{k+1}_{i+1}-\mathbf{w}^{k+1}_{i-1}}{2\Delta x}+\theta\frac{\mathbf{w}^k_{i+1}-\mathbf{w}^k_{i-1}}{2\Delta x}\right)$$
Setting $c=\frac{\Delta t}{2\Delta x}$ for conciseness and rearranging, we arrive at:
$$
\mathbf{w}^{k+1}_i+(1-\theta)c\underline{\mathbf{A}}(\mathbf{w}^{k+1}_{i+1}-\mathbf{w}^{k+1}_{i-1})
\approx\mathbf{w}^k_i-\theta c\underline{\mathbf{A}}(\mathbf{w}^k_{i+1}-\mathbf{w}^k_{i-1})
$$

This gives rise to the block matrix system
$$
\begin{bmatrix}
\underline{\mathbf{I}} & (1-\theta)c\underline{\mathbf{A}} & & & (\theta-1)c\underline{\mathbf{A}}\\
(\theta-1)c\underline{\mathbf{A}} & \underline{\mathbf{I}} & (1-\theta)c\underline{\mathbf{A}}\\
& (\theta-1)c\underline{\mathbf{A}} & \underline{\mathbf{I}} & (1-\theta)c\underline{\mathbf{A}}\\
& & \ddots & \ddots & \ddots\\
(1-\theta)c\underline{\mathbf{A}} & & & (\theta-1)c\underline{\mathbf{A}} & \underline{\mathbf{I}}\\
\end{bmatrix}\mathbf{w}^{k+1}=
\begin{bmatrix}
\underline{\mathbf{I}} & -\theta c\underline{\mathbf{A}} & & & \theta c\underline{\mathbf{A}}\\
\theta c\underline{\mathbf{A}} & \underline{\mathbf{I}} & -\theta c\underline{\mathbf{A}}\\
& \theta c\underline{\mathbf{A}} & \underline{\mathbf{I}} & -\theta c\underline{\mathbf{A}}\\
& & \ddots & \ddots & \ddots\\
-\theta c\underline{\mathbf{A}} & & & \theta c\underline{\mathbf{A}} & \underline{\mathbf{I}}\\
\end{bmatrix}\mathbf{w}^k,
$$
where $\underline{\mathbf{I}}$ is the $2\times2$ identity matrix.
Both block matrices are tridiagonal except that they have additional nonzero entries in the top-right and bottom-left entries due to the periodic boundary conditions.

The matrices are fixed in time so can be precomputed ahead-of-time.
Given the state at timestep $k$, this is a linear system we can solve to approximate the state at timestep $k+1$.

In [ ]:
%matplotlib inline

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

Set the spatial and temporal extents.

In [ ]:
X = 400
T = 4200

Define the number of points for discretising both space and time.

In [ ]:
nx = 400
nt = 4200

Determine the grid spacing, timestep, and the value of $c$.

In [ ]:
x = np.linspace(0, X, nx)
t = np.linspace(0, T, nt)
dx = x[1] - x[0]
dt = t[1] - t[0]
print(f"dx = {dx:.4f}")
print(f"dt = {dt:.4f}")
c = dt / (2 * dx)
print(f"c = {c:.4f}")

Set the $\theta$ parameter defining the timestepping scheme.

In [ ]:
# theta = jnp.array([0.0])
theta = jnp.array([0.5])
# theta = jnp.array([1.0])

In [ ]:
u0 = jnp.zeros_like(x)
eta0 = jnp.maximum(0.4 - ((x - 125.0) / 25.0) ** 2, 0.0)
w0 = jnp.vstack((u0, eta0)).transpose().flatten()

In [ ]:
b = jnp.where(x <= 50.0, 100.0, 4000.0)
g = 9.81

In [ ]:
def plot_solution(sol, axes=None, both=False):
    """Plot the solution field.

    :arg sol: the solution array corresponding to both velocity and elevation at a time level
    :kwarg axes: optional matplotlib axes object to plot on
    :kwarg both: logical flag for plotting both solution fields when True or just elevation when False
    """
    if axes is None:
        fig, axes = plt.subplots(figsize=(6, 2))
    if both:
        axes.plot(x, sol[0::2], label=r"Velocity, $u$")
    axes.plot(x, sol[1::2], label=r"Elevation, $\eta$")
    axes.set_xlabel(r"$x$ [m]")
    if both:
        axes.legend()
        axes.set_ylabel("Magnitude")
    else:
        axes.set_ylabel(r"Elevation, $\eta$ [m]")
        axes.set_ylim([-0.4, 0.4])
    axes.grid()

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 2))
plot_solution(w0, axes=axes[0], both=True)
axes[0].set_title("Initial conditions")
axes[1].set_title("Bathymetry")
axes[1].plot(x, b)
axes[1].set_xlabel(r"$x$ [m]")
axes[1].set_ylabel(r"Bathymetry, $b$ [m]")
axes[1].grid()

In [ ]:
diagonal = jnp.eye(2 * nx)
upper = jnp.diag(jnp.concatenate((jnp.vstack((jnp.zeros(nx-1), b[:-1])).transpose().flatten(), jnp.zeros(1))), k=1) \
        + jnp.diag(jnp.concatenate((jnp.vstack((g * jnp.ones(nx-2), jnp.zeros(nx-2))).transpose().flatten(), g * jnp.ones(1))), k=3) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), g * jnp.ones(1), jnp.zeros(1))), k=-(2*nx-3)) \
        + jnp.diag(b[-1:], k=-(2*nx-1))
lower = jnp.diag(jnp.concatenate((jnp.vstack((b[2:], jnp.zeros(nx-2))).transpose().flatten(), b[1:2])), k=-3) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), jnp.vstack((g * jnp.ones(nx-1), jnp.zeros(nx-1))).transpose().flatten())), k=-1) \
        + jnp.diag(jnp.concatenate((jnp.zeros(1), b[0:1], jnp.zeros(1))), k=2*nx-3) \
        + jnp.diag(g * jnp.ones(1), k=2*nx-1)

In [ ]:
# def print_matrix(mat):
#     for j, row in enumerate(mat):
#         to_print = ""
#         if j % 2 == 0:
#             print("-" * nx * 10 * 2)
#         for i, entry in enumerate(row):
#             if i % 2 == 0:
#                 to_print += "| "
#             to_print += f"{entry:8.0f} "
#         to_print += "| "
#         print(to_print)
#     print("-" * nx * 10 * 2)

$$\underline{\mathbf{A}}=\begin{bmatrix}0&g\\b&0\end{bmatrix}$$

In [ ]:
lhs_matrix = (theta - 1.0) * c * lower + diagonal + (1.0 - theta) * c * upper
rhs_matrix = theta * c * lower + diagonal - theta * c * upper

In [ ]:
fig, axes = plt.subplots(ncols=2, figsize=(12, 5))
axes[0].spy(lhs_matrix)
axes[1].spy(rhs_matrix)

In [ ]:
if np.isclose(theta[0], 0.0):
    assert np.allclose(jnp.diag(rhs_matrix), 1.0)
elif np.isclose(theta[0], 1.0):
    assert np.allclose(jnp.diag(lhs_matrix), 1.0)
else:
    print(f"No check for theta={theta[0]:.2f}")

In [ ]:
# DEBUGGING
# w = w0
plot_solution(jnp.linalg.solve(lhs_matrix, rhs_matrix @ w) - w, both=True)

In [ ]:
# Prepare axes for plotting
fig, axes = plt.subplots(nrows=5, figsize=(6, 12))
idx = 0

# Set initial condition
w = w0

# Do the time integration
for time in t:
    w = jnp.linalg.solve(lhs_matrix, rhs_matrix @ w)

    # Plot the solution field at a subset of the timesteps
    if np.isclose(time, (525, 1365, 2772, 3255, 4200)).any():
        print(f"time: {time:.0f} seconds")
        plot_solution(w, axes=axes[idx])
        # TODO: Annotate with time
        idx += 1
        
    # TODO: Progress bar

# TODO: Animation

## References

[1] Davis, B. N., & LeVeque, R. J. (2016). Adjoint methods for guiding adaptive mesh refinement in tsunami modeling. In Global tsunami science: Past and future, volume I (pp. 4055-4074). Birkhäuser, Cham.